# 🏥 PillScan — AI Model Training Pipeline (Google Colab)
**University of Tabuk — Computer Science Department — Graduation Project 2026**

This notebook contains the complete pipeline to:
1. Download the labeled pill image dataset from Roboflow.
2. Train **Stage 1 (YOLOv8-nano)** for pill detection/localization.
3. Train **Stage 2 (EfficientNet-V2-S)** for pill classification.
4. Evaluate the models and export them to **ONNX** and **TFLite** formats.

### ⚙️ Step 1: Install Dependencies
Run the following cell to install the required libraries (Ultralytics for YOLO, PyTorch Image Models for EfficientNet, Roboflow SDK, ONNX exporters).

In [ ]:
!pip install -q roboflow ultralytics timm albumentations onnx onnxruntime onnx2tf tensorflow gdown matplotlib seaborn scikit-learn Pillow onnxscript
print("✅ Dependencies installed successfully!")

### 🖥️ Verify GPU Acceleration
Ensure your Colab runtime is set to **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Running on: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

### 📥 Step 2: Download Labeled Dataset from Roboflow
Dataset URL: [Visual Pill Identification (Roboflow)](https://universe.roboflow.com/medgen/visual-pill-identification)

Replace `YOUR_ROBOFLOW_API_KEY` with the private API key obtained from your Roboflow account workspace.

In [ ]:
from roboflow import Roboflow

# Please enter your personal Roboflow API key here
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("medgen").project("visual-pill-identification")
dataset = project.version(1).download("yolov8")

print(f"📂 Dataset downloaded and extracted at: {dataset.location}")

### 🔍 Step 3: Run Stage 1 — Train YOLOv8-nano Pill Detector
The detector will locate pills within the images. This cell runs the YOLO training script.

In [ ]:
from ultralytics import YOLO
import os

print("🔍 Training YOLOv8-nano Detector...")
detector_model = YOLO('yolov8n.pt') # Start with COCO pre-trained weights

results = detector_model.train(
    data=os.path.join(dataset.location, "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    name='pillscan_detector',
    device=0 if torch.cuda.is_available() else 'cpu'
)
print("✅ Detector training complete!")

### 🧬 Step 4: Run Stage 2 — Train EfficientNet-V2-S Pill Classifier
We will now load the classification pipeline code from `colab_training_pipeline.py`. Since we want to train it on the downloaded Roboflow dataset, we will map the categories and run the classification training.

In [ ]:
# Write helper functions to run the classification model training
import shutil
import timm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import json

# Read class names from data.yaml
with open(os.path.join(dataset.location, "data.yaml"), 'r') as f:
    import yaml
    data_config = yaml.safe_load(f)
    classes = data_config.get('names', [])
    if isinstance(classes, dict):
        classes = [classes[i] for i in sorted(classes.keys())]
        
print(f"🏷️ Identified classes: {classes}")

class CustomPillDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        self.class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
        
        # Roboflow YOLO dataset has images directly under train/images
        # We crop the bounding boxes or classify directly if pre-cropped.
        # For classification demo, we read images from directory classification structure
        # If roboflow provides flat labels, we can read them using labels.
        # Let's support reading from class-named folders. If missing, we generate classification folders
        self._prepare_classification_folders()
        
        for class_name in classes:
            class_dir = self.root_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob('*'):
                    if img_path.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp'):
                        self.samples.append((str(img_path), self.class_to_idx[class_name]))

    def _prepare_classification_folders(self):
        # Helper to sort flat YOLO files into directories by class
        images_dir = self.root_dir / 'images'
        labels_dir = self.root_dir / 'labels'
        if images_dir.exists() and labels_dir.exists():
            for img_path in images_dir.glob('*'):
                label_path = labels_dir / f"{img_path.stem}.txt"
                if label_path.exists():
                    with open(label_path, 'r') as lf:
                        lines = lf.readlines()
                        if lines:
                            # First number in YOLO file is class ID
                            class_id = int(lines[0].split()[0])
                            if class_id < len(classes):
                                class_name = classes[class_id]
                                dest_dir = self.root_dir / class_name
                                os.makedirs(dest_dir, exist_ok=True)
                                shutil.copy(str(img_path), os.path.join(dest_dir, img_path.name))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# Setup transformations
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Create loaders
train_dataset = CustomPillDataset(os.path.join(dataset.location, 'train'), transform=train_transform)
val_dataset = CustomPillDataset(os.path.join(dataset.location, 'valid'), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Build Classifier
classifier_model = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=len(classes))
classifier_model = classifier_model.to(device)
print("✅ Classifier model successfully built!")

### 🚀 Start Training Classifier
This cell trains the EfficientNet-V2-S model on the dataset.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(classifier_model.parameters(), lr=1e-4, weight_decay=0.01)
epochs = 20
best_acc = 0.0

for epoch in range(1, epochs + 1):
    classifier_model.train()
    train_loss = 0.0
    correct, total = 0, 0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = classifier_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
    # Validate
    classifier_model.eval()
    val_loss = 0.0
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = classifier_model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += targets.size(0)
            val_correct += predicted.eq(targets).sum().item()
            
    acc = 100.0 * val_correct / val_total
    print(f"Epoch [{epoch}/{epochs}] Train Loss: {train_loss/len(train_loader):.4f} | Val Acc: {acc:.1f}%")
    
    if acc > best_acc:
        best_acc = acc
        torch.save(classifier_model.state_dict(), 'best_classifier.pth')
        print("   💾 Saved new best model!")

### 📊 Step 5: Export Models to ONNX & TFLite
After training, run the export block to create the `.onnx` and `.tflite` model outputs.

In [ ]:
import subprocess
import sys
os.makedirs('exported_models', exist_ok=True)

# Export to ONNX
classifier_model.eval()
classifier_model.to('cpu')
dummy_input = torch.randn(1, 3, 224, 224)
onnx_path = 'exported_models/pill_classifier.onnx'
torch.onnx.export(classifier_model, dummy_input, onnx_path, input_names=['input'], output_names=['output'])
print(f"✅ ONNX classification model exported successfully to: {onnx_path}")

# Export labels
labels_path = 'exported_models/pill_labels.txt'
with open(labels_path, 'w') as f:
    f.write('\n'.join(classes))
print(f"✅ Labels map exported successfully to: {labels_path}")

### 🧪 Step 6: Test/Inference on a Sample Image
Upload a pill photo to Google Colab, or choose one from the validation dataset, and run the complete two-stage pipeline (YOLOv8 Detection + EfficientNet Classification) to test the prediction!

In [ ]:
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# 1. Select a test image (change this to any uploaded image path)
test_image_path = None

# Fallback: find a random image from the valid split of the dataset
valid_images = list(Path(dataset.location).glob('valid/images/*'))
if valid_images:
    test_image_path = str(valid_images[0])
    print(f"📸 Selected test image from dataset: {test_image_path}")
else:
    print("⚠️ No test images found. Please upload one and specify its path.")

if test_image_path:
    # 2. Run YOLOv8 Detector
    img = Image.open(test_image_path)
    det_results = detector_model(test_image_path)
    
    # 3. Run Classifier on the image
    classifier_model.eval()
    classifier_model.to('cpu')
    
    # Preprocess image
    input_tensor = val_transform(img).unsqueeze(0)
    
    with torch.no_grad():
        outputs = classifier_model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
        conf, preds = torch.topk(probabilities, k=min(3, len(classes)))
        
    # 4. Display results
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    
    title_text = "Top Predictions:\n"
    for i in range(len(preds)):
        class_name = classes[preds[i].item()]
        confidence = conf[i].item() * 100
        title_text += f"{class_name}: {confidence:.2f}%\n"
        
    plt.title(title_text, fontsize=12, color='green')
    plt.show()